# 🌧️ Alerte Pluie Guinée - Solution Complète et Reproductible
**Hackathon IndabaX Guinée 2026**

Ce notebook présente la démarche complète de bout en bout pour prédire les précipitations des 24 prochaines heures en 4 classes (`aucune`, `faible`, `moderee`, `forte`) :
1. **Audit et Nettoyage des Données** (détection et correction des erreurs d'unités °F en °C, sentinelles de pression -999, zéros suspects d'humidité, normalisation robuste des 18 villes d'entraînement et 2 villes du test).
2. **Ingénierie des Variables Météorologiques & Climatologiques** (Point de rosée par formule de Magnus, déficit de pression de vapeur, projections des régimes de vent Mousson/Harmattan, dynamique saisonnière du FIT, cumuls de pluie passés strictement causaux avec `closed='left'`).
3. **Validation Spatio-Temporelle Robuste** (modélisant explicitement le test : années futures 2024/2025 et villes inédites avec purge de 24h).
4. **Modélisation Multi-Seed Régularisée** (LightGBM multi-classe optimisé pour la log loss).
5. **Calibration des Probabilités et Génération de la Soumission** (temperature scaling, clipping sécurisé à $[10^{-4}, 1 - 10^{-4}]$, somme exacte à 1.0 sur les 1 820 IDs notés).

In [ ]:
import os
import unicodedata
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss, classification_report

# Fixation de la graine aléatoire pour une reproductibilité exacte
RANDOM_SEED = 42
SEEDS = [42, 1337, 2026]
CLASSES = ["aucune", "faible", "moderee", "forte"]
TARGET_MAP = {c: i for i, c in enumerate(CLASSES)}

print("Environnement initialisé avec succès !")

## 1. Dictionnaire Géographique et Climatologique des 20 Villes Guinéennes
Les stations météo du challenge couvrent 20 villes guinéennes réparties sur les 4 régions naturelles.
Le test contient deux villes non vues à l'entraînement : **Labé** (Moyenne Guinée) et **Siguiri** (Haute Guinée).

In [ ]:
STATIONS_DATA = [
    {"location": "Conakry", "region": "Guinée Maritime", "lat": 9.5370, "lon": -13.6785, "altitude_m": 15, "distance_cote_km": 0, "pluie_annuelle_normale_mm": 3800},
    {"location": "Kamsar", "region": "Guinée Maritime", "lat": 10.6667, "lon": -14.6000, "altitude_m": 10, "distance_cote_km": 5, "pluie_annuelle_normale_mm": 2400},
    {"location": "Boké", "region": "Guinée Maritime", "lat": 10.9322, "lon": -14.2917, "altitude_m": 45, "distance_cote_km": 60, "pluie_annuelle_normale_mm": 2100},
    {"location": "Forécariah", "region": "Guinée Maritime", "lat": 9.4306, "lon": -13.0889, "altitude_m": 25, "distance_cote_km": 40, "pluie_annuelle_normale_mm": 3200},
    {"location": "Fria", "region": "Guinée Maritime", "lat": 10.3667, "lon": -13.5833, "altitude_m": 120, "distance_cote_km": 85, "pluie_annuelle_normale_mm": 2200},
    {"location": "Kindia", "region": "Guinée Maritime", "lat": 10.0569, "lon": -12.8658, "altitude_m": 410, "distance_cote_km": 110, "pluie_annuelle_normale_mm": 2050},
    {"location": "Télimélé", "region": "Guinée Maritime", "lat": 10.9000, "lon": -13.0333, "altitude_m": 780, "distance_cote_km": 140, "pluie_annuelle_normale_mm": 1750},
    {"location": "Mamou", "region": "Moyenne Guinée", "lat": 10.3756, "lon": -12.0911, "altitude_m": 760, "distance_cote_km": 210, "pluie_annuelle_normale_mm": 1800},
    {"location": "Dalaba", "region": "Moyenne Guinée", "lat": 10.6833, "lon": -12.2500, "altitude_m": 1200, "distance_cote_km": 200, "pluie_annuelle_normale_mm": 1650},
    {"location": "Pita", "region": "Moyenne Guinée", "lat": 11.0500, "lon": -12.4000, "altitude_m": 1000, "distance_cote_km": 200, "pluie_annuelle_normale_mm": 1600},
    {"location": "Labé", "region": "Moyenne Guinée", "lat": 11.3182, "lon": -12.2833, "altitude_m": 1020, "distance_cote_km": 215, "pluie_annuelle_normale_mm": 1550},
    {"location": "Dabola", "region": "Haute Guinée", "lat": 10.7417, "lon": -11.1083, "altitude_m": 390, "distance_cote_km": 300, "pluie_annuelle_normale_mm": 1450},
    {"location": "Faranah", "region": "Haute Guinée", "lat": 10.0411, "lon": -10.7436, "altitude_m": 450, "distance_cote_km": 330, "pluie_annuelle_normale_mm": 1700},
    {"location": "Kouroussa", "region": "Haute Guinée", "lat": 10.6500, "lon": -9.8833, "altitude_m": 360, "distance_cote_km": 420, "pluie_annuelle_normale_mm": 1400},
    {"location": "Kankan", "region": "Haute Guinée", "lat": 10.3833, "lon": -9.3000, "altitude_m": 375, "distance_cote_km": 490, "pluie_annuelle_normale_mm": 1450},
    {"location": "Siguiri", "region": "Haute Guinée", "lat": 11.4167, "lon": -9.1667, "altitude_m": 350, "distance_cote_km": 520, "pluie_annuelle_normale_mm": 1250},
    {"location": "Kissidougou", "region": "Guinée Forestière", "lat": 9.1833, "lon": -10.1000, "altitude_m": 530, "distance_cote_km": 390, "pluie_annuelle_normale_mm": 1900},
    {"location": "Guéckédou", "region": "Guinée Forestière", "lat": 8.5667, "lon": -10.1333, "altitude_m": 420, "distance_cote_km": 370, "pluie_annuelle_normale_mm": 2300},
    {"location": "Macenta", "region": "Guinée Forestière", "lat": 8.5436, "lon": -9.4711, "altitude_m": 560, "distance_cote_km": 440, "pluie_annuelle_normale_mm": 2400},
    {"location": "Nzérékoré", "region": "Guinée Forestière", "lat": 7.7561, "lon": -8.8178, "altitude_m": 480, "distance_cote_km": 500, "pluie_annuelle_normale_mm": 2100}
]
stations_df = pd.DataFrame(STATIONS_DATA)
stations_df.head(5)

## 2. Chargement et Nettoyage Robuste des Relevés Météo

In [ ]:
CITY_ALIAS_MAP = {
    "boke": "Boké", "boké": "Boké",
    "conakry": "Conakry", "conakry ": "Conakry",
    "dabola": "Dabola", "dalaba": "Dalaba", "faranah": "Faranah",
    "forecariah": "Forécariah", "forécariah": "Forécariah",
    "fria": "Fria", "gueckedou": "Guéckédou", "gueckédou": "Guéckédou", "guéckédou": "Guéckédou",
    "kamsar": "Kamsar", "kankan": "Kankan", "kankan ": "Kankan",
    "kindia": "Kindia", "kissi": "Kissidougou", "kissidougou": "Kissidougou",
    "kouroussa": "Kouroussa", "labe": "Labé", "labé": "Labé", "labe ": "Labé",
    "macenta": "Macenta", "mamou": "Mamou",
    "n'zérékoré": "Nzérékoré", "nzerekore": "Nzérékoré", "nzérékoré": "Nzérékoré",
    "pita": "Pita", "siguiri": "Siguiri", "telimele": "Télimélé", "télimélé": "Télimélé"
}

def normalize_city(raw_name):
    if not isinstance(raw_name, str):
        return "Conakry"
    k = raw_name.strip().lower()
    if k in CITY_ALIAS_MAP:
        return CITY_ALIAS_MAP[k]
    norm = unicodedata.normalize("NFKD", k).encode("ASCII", "ignore").decode("utf-8")
    return CITY_ALIAS_MAP.get(norm, raw_name.strip())

train_raw = pd.read_csv("data/train.csv")
test_raw = pd.read_csv("data/test.csv")
sub_raw = pd.read_csv("data/sample_submission.csv")

train_raw["location"] = train_raw["location"].apply(normalize_city)
test_raw["location"] = test_raw["location"].apply(normalize_city)

print(f"Villes uniques train ({len(train_raw['location'].unique())}) :", sorted(train_raw["location"].unique()))
print(f"Villes uniques test  ({len(test_raw['location'].unique())}) :", sorted(test_raw["location"].unique()))
assert len(train_raw["location"].unique()) == 18, "Train doit contenir exactement 18 villes."
assert len(test_raw["location"].unique()) == 20, "Test doit contenir exactement 20 villes."

### Nettoyage des capteurs et sentinelles

In [ ]:
def clean_weather_dataframe(df):
    d = df.copy()
    d["timestamp"] = pd.to_datetime(d["ingestion_timestamp"])
    d["year"] = d["timestamp"].dt.year
    
    # Déduplication exacte
    d = d.drop_duplicates(subset=["location", "timestamp"]).reset_index(drop=True)
    
    # Correction °F -> °C pour T > 50°C
    f_mask = d["temperature"] > 50.0
    d.loc[f_mask, "temperature"] = (d.loc[f_mask, "temperature"] - 32.0) * (5.0 / 9.0)
    
    # Sentinelles de pression
    p_mask = (d["pressure"] == -999.0) | (d["pressure"] < 500.0) | (d["pressure"] > 1100.0)
    d.loc[p_mask, "pressure"] = np.nan
    
    # Humidité suspecte = 0
    d.loc[d["humidity"] == 0.0, "humidity"] = np.nan
    
    # Bornes physiques
    d["humidity"] = d["humidity"].clip(0.0, 100.0)
    d["cloud_cover"] = d["cloud_cover"].clip(0.0, 100.0)
    d["wind_speed"] = d["wind_speed"].clip(0.0, 60.0)
    d["wind_deg"] = d["wind_deg"] % 360.0
    d["rain_3h"] = d["rain_3h"].clip(0.0, 500.0)
    
    # Tri causal chronologique par ville
    d = d.sort_values(["location", "timestamp"]).reset_index(drop=True)
    return d

train_clean = clean_weather_dataframe(train_raw)
test_clean = clean_weather_dataframe(test_raw)
print("Nettoyage terminé avec succès !")

## 3. Ingénierie des Variables Météorologiques & Climatologiques Causales

In [ ]:
from src.features import build_feature_pipeline

train_feat, test_feat, feature_cols = build_feature_pipeline(
    train_clean, test_clean, include_raw_location=False
)
print(f"Nombre de variables générées ({len(feature_cols)}) :\n", feature_cols)

## 4. Validation Spatio-Temporelle & Entraînement Multi-Seed LightGBM
On évalue le modèle selon le protocole spatio-temporel : entraînement sur les années $\le 2023$ sur des villes vues, validation sur des villes non vues en 2024 avec purge de 24h.

In [ ]:
from src.cv import spatio_temporal_folds
from src.models import get_regularized_lgb_params, train_single_lgb, predict_proba_lgb
from src.calibration import evaluate_log_loss_breakdown, fit_temperature_scaling, apply_temperature_scaling

X = train_feat[feature_cols]
y = train_feat["target"].map(TARGET_MAP).to_numpy()

folds = list(spatio_temporal_folds(train_feat, n_splits=4, random_state=RANDOM_SEED))
oof_proba = np.zeros((len(train_feat), len(CLASSES)), dtype=np.float64)
val_indices = []
fold_losses = []

for fold_idx, (tr_idx, va_idx, heldout_cities) in enumerate(folds):
    print(f"\n--- Fold {fold_idx + 1}/4 | Villes 2024 masquées : {heldout_cities} ---")
    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]
    
    params = get_regularized_lgb_params(seed=RANDOM_SEED)
    booster = train_single_lgb(
        X_tr, y_tr, X_va, y_va,
        params=params,
        num_boost_round=400,
        early_stopping_rounds=40,
        verbose_eval=100
    )
    va_pred = predict_proba_lgb(booster, X_va)
    oof_proba[va_idx] = va_pred
    val_indices.extend(va_idx)
    
    ev = evaluate_log_loss_breakdown(y_va, va_pred, title=f"Pli {fold_idx + 1}")
    fold_losses.append(ev["log_loss"])

print(f"\n>>> Résultat CV Spatio-Temporelle : Log Loss Moyenne = {np.mean(fold_losses):.5f} ± {np.std(fold_losses):.5f}")

## 5. Entraînement Final Multi-Seed et Soumission
On entraîne un ensemble de 3 graines aléatoires (42, 1337, 2026) sur l'ensemble complet des données d'entraînement pour maximiser la robustesse.

In [ ]:
import lightgbm as lgb
import xgboost as xgb

to_predict = test_feat[test_feat["ID"].isin(sub_raw["ID"])].copy().reset_index(drop=True)
X_test = to_predict[feature_cols].copy()

cat_cols = [c for c in X.columns if X[c].dtype.name == "category"]

# 1. LightGBM Multiclasse (3 graines : 42, 1337, 2026 - 220 arbres optimaux)
p_test_mc = np.zeros((len(to_predict), len(CLASSES)), dtype=np.float64)
for s in SEEDS:
    print(f"LGB Multiclasse - Seed {s}...")
    params = get_regularized_lgb_params(seed=s)
    params["learning_rate"] = 0.03
    dtr = lgb.Dataset(X, label=y, categorical_feature=cat_cols)
    clf = lgb.train(params, dtr, num_boost_round=220)
    p_test_mc += clf.predict(X_test) / len(SEEDS)

# 2. LightGBM Ordinal (3 seuils cumulatifs : >=1mm, >=10mm, >=30mm - 180 arbres)
p_test_ord = np.zeros((len(to_predict), len(CLASSES)), dtype=np.float64)
ord_seeds = [42, 1337]
for s in ord_seeds:
    print(f"LGB Ordinal - Seed {s}...")
    p_cum = np.zeros((len(to_predict), 3), dtype=np.float64)
    for t_idx, threshold in enumerate([1, 2, 3]):
        y_bin = (y >= threshold).astype(int)
        dtr = lgb.Dataset(X, label=y_bin, categorical_feature=cat_cols)
        params_bin = {
            "objective": "binary",
            "metric": "binary_logloss",
            "learning_rate": 0.03,
            "num_leaves": 35,
            "max_depth": 6,
            "min_child_samples": 50,
            "subsample": 0.80,
            "colsample_bytree": 0.75,
            "reg_alpha": 0.2,
            "reg_lambda": 2.0,
            "verbose": -1,
            "random_state": s + threshold,
        }
        clf = lgb.train(params_bin, dtr, num_boost_round=180)
        p_cum[:, t_idx] = clf.predict(X_test)
    p1 = p_cum[:, 0]
    p2 = np.minimum(p1, p_cum[:, 1])
    p3 = np.minimum(p2, p_cum[:, 2])
    p_4c = np.column_stack([
        np.clip(1.0 - p1, 0.0, 1.0),
        np.clip(p1 - p2, 0.0, 1.0),
        np.clip(p2 - p3, 0.0, 1.0),
        np.clip(p3, 0.0, 1.0),
    ])
    p_test_ord += safe_clip_and_renormalize(p_4c) / len(ord_seeds)

# 3. XGBoost Multiclasse (2 graines : 42, 1337 - 150 arbres)
p_test_xgb = np.zeros((len(to_predict), len(CLASSES)), dtype=np.float64)
X_xgb = X.copy()
X_test_xgb = X_test.copy()
for c in cat_cols:
    X_xgb[c] = X_xgb[c].cat.codes.astype("category")
    X_test_xgb[c] = X_test_xgb[c].cat.codes.astype("category")

for s in [42, 1337]:
    print(f"XGBoost - Seed {s}...")
    xgb_clf = xgb.XGBClassifier(
        n_estimators=150, learning_rate=0.04, max_depth=5, subsample=0.80,
        colsample_bytree=0.75, reg_alpha=0.3, reg_lambda=2.0,
        objective="multi:softprob", num_class=4, tree_method="hist",
        enable_categorical=True, random_state=s, n_jobs=-1
    )
    xgb_clf.fit(X_xgb, y, verbose=False)
    p_test_xgb += xgb_clf.predict_proba(X_test_xgb) / 2.0

# 4. Blend Triple : 55% LGB MC + 30% LGB Ordinal + 15% XGBoost
p_blend = 0.55 * p_test_mc + 0.30 * p_test_ord + 0.15 * p_test_xgb

# 5. Calibration de température T=1.02 et clipping sécurisé [1e-4, 1-1e-4]
calibrated_preds = apply_temperature_scaling(p_blend, temperature=1.02)

# 6. Construction et vérification de la soumission
submission = pd.DataFrame(calibrated_preds, columns=CLASSES)
submission.insert(0, "ID", to_predict["ID"].values)
final_submission = sub_raw[["ID"]].merge(submission, on="ID", how="left")

assert len(final_submission) == 1820
assert list(final_submission.columns) == ["ID"] + CLASSES
assert (final_submission["ID"].values == sub_raw["ID"].values).all()
assert not final_submission[CLASSES].isna().any().any()
assert (np.abs(final_submission[CLASSES].sum(axis=1) - 1.0).max()) < 1e-4

os.makedirs("submissions", exist_ok=True)
final_submission.to_csv("submissions/submission_notebook_reproductible.csv", index=False)
print("\nSoumission générée avec succès : submissions/submission_notebook_reproductible.csv")
display(final_submission.head(10))
